# Fintual: Fincard
Integrantes: Iair Chistik, Francisco Cisternas y Agustin rosales

In [2]:
# Librerias 
import pandas as pd 
import numpy as numpy
import matplotlib.pyplot as plr
import seaborn as sns

# Configuraciones de visualización
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")

In [3]:
# Carga de los datos inicales
diccionario = pd.read_csv("diccionario_datos_2026.csv")
fintual_percepcion = pd.read_csv("fintual_perception_dataset_2026.csv")
usuarios = pd.read_csv("fintual_users_dataset_2026.csv")
caracteristicas = pd.read_csv("user_demographics_2026.csv")

### EDA DE DIAGNOSTICO
El objetivo de realizar un EDA en esta fase es diagnosticar el estado en el que se encuentran las bases de datos para encontrar anomalias como datos fuera de rango, valores nulos, duplicados, errores de digitacio, etc. Los objetivos son:
- Dimensiones
- Valores duplicados
- Valores nulos 
- Anomalias de digitación 
- Valores fuera de rango 
- Valores atipicos

In [73]:
# Inspección de los tablas (Dimensiones)
base_datos = {
    "usuarios": usuarios,
    "caracteristicas": caracteristicas,
    "fintual_percepcion": fintual_percepcion
}
for nombre, tablas in base_datos.items():
    filas, columnas = tablas.shape
    print(f"El dataset '{nombre}' tiene {filas} filas y {columnas} columnas.")

El dataset 'usuarios' tiene 12025 filas y 6 columnas.
El dataset 'caracteristicas' tiene 12000 filas y 11 columnas.
El dataset 'fintual_percepcion' tiene 21000 filas y 12 columnas.


In [ ]:
# Usuarios unicos 
usuarios_unicos = usuarios['user_id'].nunique()
print(f"Usuarios unicos: {usuarios_unicos}")
usuarios_unicos_caracteristicas = caracteristicas['user_id'].nunique()
print(f"Usuarios unicos en caracteristicas: {usuarios_unicos_caracteristicas}")
usuarios_unicos_percepcion = fintual_percepcion['user_id'].nunique()
print(f"Usuarios unicos en percepcion: {usuarios_unicos_percepcion}")

Usuarios unicos: 12000
Usuarios unicos en caracteristicas: 12000
Usuarios unicos en percepcion: 3000


In [101]:
# Estadisticos usuarios
usuarios.describe().T

,count,mean,std,min,25%,50%,75%,max
age,"11,840.00",33.61,10.17,0.00,25.00,33.00,40.00,150.00
total_invested,"12,025.00","8,106,127.96","13,929,064.99","15,500.00","302,300.00","2,824,300.00","9,637,000.00","239,386,000.00"
risk_tolerance,"11,647.00",3.22,1.21,0.00,2.00,3.00,4.00,5.00
months_active,"12,025.00",32.54,22.28,1.00,12.00,30.00,50.00,96.00
app_sessions_month,"11,581.00",20.97,10.01,1.00,13.00,21.00,29.00,47.00


In [98]:
# Estadisticos caracteristicas
caracteristicas.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
user_id,12000,12000,FTL_00001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
region,11881,16,Metropolitana de Santiago,6830,NaN,NaN,NaN,NaN,NaN,NaN,NaN
genero,11772,3,Masculino,6204,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tramo_ingreso,11009,5,$1.200.001 - $2.500.000,3059,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ocupacion,12000,5,Dependiente,7707,NaN,NaN,NaN,NaN,NaN,NaN,NaN
banco_principal,12000,12,Banco de Chile,2340,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tiene_tarjeta_credito,12000,2,Sí,7989,NaN,NaN,NaN,NaN,NaN,NaN,NaN
n_tarjetas_credito,"12,000.00",NaN,NaN,NaN,1.20,1.07,0.00,0.00,1.00,2.00,4.00
billetera_digital_principal,12000,5,Ninguna,4444,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gasto_mensual_tarjetas_clp,"12,000.00",NaN,NaN,NaN,"1,155,643.58","1,046,950.30","48,000.00","440,000.00","848,000.00","1,519,000.00","14,602,000.00"


In [104]:
# Estadisticos percepcion
fintual_percepcion.describe().T

,count,mean,std,min,25%,50%,75%,max
ease_of_use,"20,202.00",5.57,1.17,1.00,5.00,6.00,7.00,7.00
fees_transparency,"20,016.00",4.75,1.39,1.00,4.00,5.00,6.00,7.00
investment_options,"19,882.00",4.54,1.44,1.00,4.00,5.00,6.00,7.00
customer_support,"19,779.00",4.68,1.18,1.00,4.00,5.00,5.00,7.00
mobile_experience,"20,168.00",5.63,1.15,1.00,5.00,6.00,7.00,7.00
educational_content,"20,029.00",4.26,1.40,1.00,3.00,4.00,5.00,7.00
trust_security,"20,228.00",5.27,1.13,1.00,5.00,5.00,6.00,7.00
returns_performance,"19,521.00",4.67,1.23,1.00,4.00,5.00,6.00,7.00
social_impact,"19,360.00",4.28,1.21,1.00,3.00,4.00,5.00,7.00
innovation,"20,020.00",5.20,1.35,1.00,4.00,5.00,6.00,7.00


In [78]:
# Detalle especifico de los nulos por base y columna
def resumen_nulos(diccionario_dfs):
    for nombre, df in diccionario_dfs.items():
        conteo_nulos = df.isnull().sum()
        porcentaje_nulos = (df.isnull().mean() * 100).round(2)
        
        # Armamos el DataFrame resumen
        resumen = pd.DataFrame({
            "Nulos": conteo_nulos,
            "% Nulos": porcentaje_nulos
        })
        
        print(f"-------- Nulos en '{nombre}' (Total filas: {len(df)}) --------")
        display(resumen)

resumen_nulos(base_datos)


-------- Nulos en 'usuarios' (Total filas: 12025) --------


,Nulos,% Nulos
user_id,0,0.00
age,185,1.54
total_invested,0,0.00
risk_tolerance,378,3.14
months_active,0,0.00
app_sessions_month,444,3.69


-------- Nulos en 'caracteristicas' (Total filas: 12000) --------


,Nulos,% Nulos
user_id,0,0.00
region,119,0.99
genero,228,1.90
tramo_ingreso,991,8.26
ocupacion,0,0.00
banco_principal,0,0.00
tiene_tarjeta_credito,0,0.00
n_tarjetas_credito,0,0.00
billetera_digital_principal,0,0.00
gasto_mensual_tarjetas_clp,0,0.00


-------- Nulos en 'fintual_percepcion' (Total filas: 21000) --------


,Nulos,% Nulos
user_id,0,0.00
brand,0,0.00
ease_of_use,798,3.80
fees_transparency,984,4.69
investment_options,1118,5.32
customer_support,1221,5.81
mobile_experience,832,3.96
educational_content,971,4.62
trust_security,772,3.68
returns_performance,1479,7.04


In [116]:
# Edades registradas que están fuera de 18 a 110
edades_invalidas = usuarios[usuarios['age'].notnull() & ~usuarios['age'].between(18, 110)]
print(f"Registros con edad inválida/imposible: {len(edades_invalidas)}")
display(edades_invalidas[['user_id', 'age']])
# Tolerancia de riesgo fuera de 1 a 5
tolerancia_invalidas = usuarios[usuarios['risk_tolerance'].notnull() & ~usuarios['risk_tolerance'].between(1, 5)]
print(f"Registros con tolerancia de riesgo inválida: {len(tolerancia_invalidas)}")
display(tolerancia_invalidas[['user_id', 'risk_tolerance']])

Registros con edad inválida/imposible: 15


,user_id,age
931,FTL_00930,17.00
1383,FTL_01382,150.00
1578,FTL_01576,0.00
2206,FTL_02203,17.00
2545,FTL_02540,17.00
3519,FTL_03510,17.00
4190,FTL_04179,17.00
4197,FTL_04186,7.00
5316,FTL_05303,3.00
5457,FTL_05443,17.00


Registros con tolerancia de riesgo inválida: 70


,user_id,risk_tolerance
240,FTL_00241,0.00
388,FTL_00389,0.00
987,FTL_00986,0.00
998,FTL_00997,0.00
1154,FTL_01153,0.00
...,...,...
10853,FTL_10831,0.00
10874,FTL_10852,0.00
11057,FTL_11035,0.00
11170,FTL_11148,0.00
